In [1]:
# %pip install mlflow

In [29]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import mlflow
import mlflow.sklearn

<h2>Read the datafile</h2>

In [3]:
df = pd.read_parquet("data/processed/taxi_demand_features.parquet")
df.head()

,pickup_hour,PULocationID,number_pickups,hour,day_of_week,previous_hour,previous_day,previous_week,rolling_24h,rolling_3h
0,2026-01-08 00:00:00,1,0,0,0,0.0,0.0,0.0,0.25,0.0
1,2026-01-08 01:00:00,1,0,0,0,0.0,0.0,0.0,0.25,0.0
2,2026-01-08 02:00:00,1,0,0,0,0.0,0.0,1.0,0.25,0.0
3,2026-01-08 03:00:00,1,0,0,0,0.0,0.0,0.0,0.25,0.0
4,2026-01-08 04:00:00,1,0,0,0,0.0,0.0,3.0,0.25,0.0


## Train-test split

we make dates before 26th jan training samples and after that testing samples. kinda wanna predict future from the past

In [4]:
print(df["pickup_hour"].min())
print(df["pickup_hour"].max())

2026-01-08 00:00:00
2026-01-31 23:00:00


In [5]:
train_df = df[df['pickup_hour'] < "2026-01-26"].copy()
test_df = df[df['pickup_hour'] > "2026-01-26"].copy()

In [6]:
print(train_df["pickup_hour"].min())
print(train_df["pickup_hour"].max())

print(test_df["pickup_hour"].min())
print(test_df["pickup_hour"].max())

print(train_df.shape)
print(test_df.shape)

2026-01-08 00:00:00
2026-01-25 23:00:00
2026-01-26 01:00:00
2026-01-31 23:00:00
(113184, 10)
(37466, 10)


In [7]:
features = ["PULocationID", "hour", "day_of_week", "previous_hour", "previous_day", "previous_week",
           "rolling_24h", "rolling_3h"]
X_train = train_df[features]
y_train = train_df["number_pickups"]


X_test = test_df[features]
y_test = test_df["number_pickups"]
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(113184, 8)
(113184,)
(37466, 8)
(37466,)


<h1> Baseline Model (Random Forest Regressor)</h1>

In [8]:
model = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42, 
    n_jobs = -1
)

model.fit(X_train, y_train)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [9]:
y_pred = model.predict(X_test)

In [10]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 3.4879594759896477
RMSE: 9.539605789678305


If we predict current hour demand based on just the previour hour

In [11]:
base_pred =  test_df['previous_hour']

base_mae = mean_absolute_error(y_test, base_pred)
base_rmse = np.sqrt(mean_squared_error(y_test, base_pred))

print("Baseline MAE:", base_mae)
print("Baseline RMSE:", base_rmse)

Baseline MAE: 5.50280254097048
Baseline RMSE: 15.102291829104358


## Insights into where the model is making mistakes

In [12]:
results = test_df.copy()

results["predicted"] = y_pred
results["error"] = results["number_pickups"] - results["predicted"]
results["absolute_error"] = results["error"].apply(lambda x: abs(x))

results["absolute_error"].describe()

count    37466.000000
mean         3.487959
std          8.879207
min          0.000000
25%          0.000000
50%          0.690000
75%          2.750000
max        151.230000
Name: absolute_error, dtype: float64

In [13]:
results.sort_values(by = "absolute_error", ascending = False).head(10)

,pickup_hour,PULocationID,number_pickups,hour,day_of_week,previous_hour,previous_day,previous_week,rolling_24h,rolling_3h,predicted,error,absolute_error
80015,2026-01-29 23:00:00,142,146,23,3,344.0,195.0,393.0,182.958333,385.666667,297.23,-151.23,151.23
134109,2026-01-27 21:00:00,236,299,21,1,236.0,80.0,139.0,228.333333,364.000000,149.16,149.84,149.84
130654,2026-01-27 22:00:00,230,278,22,1,404.0,113.0,364.0,110.416667,321.333333,427.21,-149.21,149.21
105285,2026-01-26 21:00:00,186,310,21,0,150.0,14.0,196.0,48.708333,135.666667,163.68,146.32,146.32
77655,2026-01-27 15:00:00,138,140,15,1,271.0,137.0,268.0,131.583333,187.000000,280.09,-140.09,140.09
74181,2026-01-26 21:00:00,132,401,21,0,435.0,2.0,500.0,128.208333,377.666667,540.47,-139.47,139.47
90909,2026-01-27 21:00:00,161,561,21,1,594.0,247.0,653.0,189.958333,536.333333,697.38,-136.38,136.38
74180,2026-01-26 20:00:00,132,435,20,0,304.0,5.0,505.0,110.291667,386.666667,298.92,136.08,136.08
90933,2026-01-28 21:00:00,161,553,21,2,595.0,561.0,665.0,234.333333,532.333333,687.17,-134.17,134.17
80013,2026-01-29 21:00:00,142,505,21,3,308.0,362.0,483.0,175.166667,291.333333,374.74,130.26,130.26


In [14]:
#how often errors above threshold occur
for threshold in [5, 10, 20, 50, 100]:
    pct = (results["absolute_error"] > threshold).mean() * 100
    print(f"Error : {pct}, Threshold :{ threshold}")
    

Error : 16.062563390807664, Threshold :5
Error : 8.799978647306892, Threshold :10
Error : 4.057011690599477, Threshold :20
Error : 0.7420060855175359, Threshold :50
Error : 0.09074894571077777, Threshold :100


DtreeRegression performs better than baseline prediction however, most extreme errors lie at the tail of the distribution

In [15]:
r2 = r2_score(y_test, y_pred)
print(f"R2: {r2}")

R2: 0.970493077454245


Model explains about 97 variance in the test set number of pickups prediction.

## Model tracking via mlflow

In [16]:
experiment_name = "nyc_taxi_demand"
uri = "sqlite:///C:/Users/aahan/OneDrive/Desktop/Machine%20Learning/Mlops/mlflow.db"

mlflow.set_tracking_uri(
    uri
)

if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(experiment_name)
    
mlflow.set_experiment(experiment_name)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name(experiment_name))


Tracking URI: sqlite:///C:/Users/aahan/OneDrive/Desktop/Machine%20Learning/Mlops/mlflow.db
Experiment: <Experiment: artifact_location='file:///C:/Users/aahan/OneDrive/Desktop/Machine Learning/Mlops/mlruns/1', creation_time=1789154952172, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789154952172, lifecycle_stage='active', name='nyc_taxi_demand', tags={}, trace_location=None, workspace='default'>


In [38]:
def mlflowrunRf(n_estimators = 100, model_path = "model"):
    with mlflow.start_run():
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_test)
    
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
    
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("random_state", 42)
    
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)

        #log the model
        mlflow.sklearn.log_model(
            sk_model = model,
            name = model_path
        )
    
        print("MAE:", mae)
        print("RMSE:", rmse)
        print("R2:", r2)
    


    

In [39]:
model_path = "random_forest_taxi_demand"
mlflowrunRf(100, model_path)

MAE: 3.4879594759896477
RMSE: 9.539605789678305
R2: 0.970493077454245


In [40]:

print(mlflow.search_runs())


                             run_id experiment_id    status  \
0  4d351262fd484bdca517eeb4fa17018a             1  FINISHED   

                                        artifact_uri  \
0  file:///C:/Users/aahan/OneDrive/Desktop/Machin...   

                        start_time                         end_time  \
0 2026-09-11 20:00:05.719000+00:00 2026-09-11 20:00:51.687000+00:00   

   metrics.r2  metrics.mae  metrics.rmse params.random_state  \
0    0.970493     3.487959      9.539606                  42   

  params.n_estimators tags.mlflow.source.type tags.mlflow.runName  \
0                 100                NOTEBOOK       wise-fawn-119   

  tags.mlflow.user tags.mlflow.source.name  
0            aahan    Model_training.ipynb  


In [20]:
import subprocess

subprocess.Popen([
    sys.executable,
    "-m",
    "mlflow",
    "ui",
    "--backend-store-uri",
    uri,
    "--port",
    "5000"
])

print("MLflow UI started")

MLflow UI started


## Second Run

In [42]:
mlflowrunRf(200, model_path)

MAE: 3.4639670307300894
RMSE: 9.469094849197516
R2: 0.9709276597184231


## Model Registry (official version of model, to deploy)

In [75]:

model_name = "random_forest_taxi_demand"
model_id = "m-fc6d80ace64a452ab46fdc4fdaa50051"

def register_model(model_name : str, model_id : str):
    model_uri = f"models:/{model_id}"
    registered_model = mlflow.register_model(model_uri = model_uri, name = model_name)
    return registered_model
    


In [76]:
r_m = register_model(model_name, model_id)
print(r_m)

<ModelVersion: aliases=[], creation_timestamp=1789158547258, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1789158547258, metrics=None, model_id=None, name='random_forest_taxi_demand', params=None, run_id='017a6f51c77347139767b73b302f0bc0', run_link=None, source='models:/m-fc6d80ace64a452ab46fdc4fdaa50051', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>


Successfully registered model 'random_forest_taxi_demand'.
Created version '1' of model 'random_forest_taxi_demand'.


In [77]:
# client = mlflow.MlflowClient()
# client.delete_registered_model(name="random_forest_taxi_demand_1")

In [78]:
# run_id = "017a6f51c77347139767b73b302f0bc0"

# logged_models = mlflow.search_logged_models(
#     filter_string=f"source_run_id = '{run_id}'"
# )

# print(logged_models)

## Registering another model

In [79]:
model_path = "random_forest_taxi_demand"
mlflowrunRf(300, model_path)

MAE: 3.457842225255722
RMSE: 9.441389855597677
R2: 0.971097532490734


In [80]:
model_name = "random_forest_taxi_demand"
model_id = "m-9e484644ef444ac186dac76b1e1df70f"

r_m = register_model(model_name, model_id)
print(r_m)

<ModelVersion: aliases=[], creation_timestamp=1789158990657, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1789158990657, metrics=None, model_id=None, name='random_forest_taxi_demand', params=None, run_id='d6effcc7cb1545afbc0a7d971a3dcd2d', run_link=None, source='models:/m-9e484644ef444ac186dac76b1e1df70f', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>


Registered model 'random_forest_taxi_demand' already exists. Creating a new version of this model...
Created version '2' of model 'random_forest_taxi_demand'.


In [82]:
m_r = mlflow.pyfunc.load_model("models:/random_forest_taxi_demand/2")

In [84]:
y_pred_r = m_r.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_r)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_r))
r2 = r2_score(y_test, y_pred_r)
        
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
    


MAE: 3.457842225255722
RMSE: 9.441389855597677
R2: 0.971097532490734
